# 👔 Virtual Stylist — Siamese Compatibility Network

> **✅ HỖ TRỢ CHECKPOINT — Có thể tiếp tục từ bất kỳ tài khoản Colab nào!**
>
> Mỗi bước sẽ tự động lưu kết quả lên Google Drive trước khi tiếp tục bước tiếp theo.
> Nếu phải đổi tài khoản, chỉ cần chạy lại từ đầu — mọi bước đã xong sẽ được **bỏ qua tự động**.

| Bước | Nội dung |
|------|----------|
| ✅ 0 | Mount Google Drive & Cấu hình đường dẫn Checkpoint |
| ✅ 1 | Cài đặt thư viện |
| ✅ 2 | Thiết lập Thư mục Làm việc |
| ✅ 3 | Kiểm tra Dataset trên Drive |
| ✅ 4 | Xây dựng Positive Pairs (có checkpoint) |
| ✅ 5 | Negative Sampling nhanh (có checkpoint) |
| ✅ 6 | Extract CLIP Embeddings (có checkpoint) |
| ✅ 7 | Huấn luyện Siamese Model (có checkpoint epoch) |
| ✅ 8 | Demo Inference: Phối đồ thực tế |
| ✅ 9 | Lưu Model Artifacts |

> **Runtime → Change runtime type → T4 GPU** trước khi chạy!


## 💾 Bước 0 — Mount Google Drive (Checkpoint Storage)
**Chạy ô này TRƯỚC TIÊN** — mọi checkpoint sẽ được lưu vào Drive để có thể tiếp tục từ tài khoản khác.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Thư mục checkpoint trên Drive ─────────────────────────────────────────
DRIVE_DIR   = '/content/drive/MyDrive/Virtual_Stylist_application'
os.chdir(DRIVE_DIR)
CKPT_DIR    = os.path.join(DRIVE_DIR, 'checkpoints')
OUTPUTS_DIR = '/content/outputs'

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# ── Helpers ────────────────────────────────────────────────────────────────
def ckpt(name): return os.path.join(CKPT_DIR, name)
def done(name): return os.path.exists(ckpt(name))
def skip(name): print(f'⏭️  [{name}] đã có trên Drive — bỏ qua!')

print('✅ Google Drive đã mount thành công!')
print(f'📂 Checkpoint dir: {CKPT_DIR}')
print('\nCheckpoints hiện có trên Drive:')
for f in sorted(os.listdir(CKPT_DIR)):
    mb = os.path.getsize(ckpt(f)) / 1e6
    print(f'   ✅  {f:40s}  {mb:.1f} MB')


## ⚙️ Bước 1 — Cài đặt thư viện

In [ ]:
!pip install -q transformers torch torchvision pillow\
             kaggle pandas numpy scikit-learn matplotlib\
             seaborn tqdm faiss-cpu
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 📁 Bước 2 — Thiết lập Thư mục Làm việc

In [ ]:
import os
DRIVE_DIR = '/content/drive/MyDrive/Virtual_Stylist_application'
os.chdir(DRIVE_DIR)
print(f'✅ Working directory changed to: {os.getcwd()}')


## 📦 Bước 3 — Chuẩn bị Dataset (Sử dụng dữ liệu trên Drive)

Cell này sẽ kiểm tra và sử dụng trực tiếp dataset đã lưu sẵn trên Google Drive, bỏ qua bước tải từ Kaggle.

1. **CSV Files**: Kiểm tra `articles.csv` và `transactions_train.csv`.
2. **Images**: Nếu `/content/images` chưa có, sẽ tiến hành giải nén từ file `.zip` có trên Drive.


In [ ]:
import os, subprocess, time
from pathlib import Path

# ── Đường dẫn ─────────────────────────────────────────────────────────────
LOCAL_IMG_DIR  = '/content/images'
REQUIRED_CSVS  = ['articles.csv', 'transactions_train.csv']

# ══════════════════════════════════════════════════════════════════════════
# STEP A — Kiểm tra CSV trên Drive
# ══════════════════════════════════════════════════════════════════════════
print('\n🔍 Kiểm tra dữ liệu CSV trên Drive...')
missing_csvs = [f for f in REQUIRED_CSVS if not os.path.exists(os.path.join(DRIVE_DIR, f))]

if missing_csvs:
    raise FileNotFoundError(f'❌ Thiếu các file CSV trên Drive: {missing_csvs}')
else:
    print('✅ Tất cả CSV đã sẵn sàng trên Drive.')
    os.chdir(DRIVE_DIR)

# ══════════════════════════════════════════════════════════════════════════
# STEP B — Giải nén ảnh vào SSD /content/images
# ══════════════════════════════════════════════════════════════════════════
if os.path.exists(LOCAL_IMG_DIR) and len(os.listdir(LOCAL_IMG_DIR)) > 10:
    n = len(os.listdir(LOCAL_IMG_DIR))
    print(f'\n✅ [SSD Cache] /content/images đã có {n} thư mục — bỏ qua giải nén.')
else:
    zip_candidates = list(Path(DRIVE_DIR).glob('*.zip'))
    if not zip_candidates:
        raise FileNotFoundError(f'❌ Không tìm thấy file .zip nào trong {DRIVE_DIR} để giải nén ảnh!')

    src_zip = str(zip_candidates[0])
    print(f'\n⚡ Tìm thấy {os.path.basename(src_zip)} trên Drive.')
    print('⚡ Đang giải nén ảnh từ Drive vào /content/images (sẽ mất vài phút) ...')
    
    t0 = time.time()
    subprocess.run(['unzip', '-q', '-o', src_zip, 'images/*', '-d', '/content/'], check=True)
    elapsed = time.time() - t0
    
    n_img = sum(len(fs) for _, _, fs in os.walk(LOCAL_IMG_DIR)) if os.path.exists(LOCAL_IMG_DIR) else 0
    print(f'✅ Giải nén xong! {n_img:,} ảnh ({elapsed:.0f}s)')

# ══════════════════════════════════════════════════════════════════════════
# STEP C — Tổng kiểm tra
# ══════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  TỔNG KIỂM TRA DATASET')
print('='*60)
for csv_name in REQUIRED_CSVS:
    path = os.path.join(DRIVE_DIR, csv_name)
    mb = os.path.getsize(path) / 1e6
    print(f'  ✅  {csv_name:<35s} {mb:>8.1f} MB  [Drive]')

n_img = sum(len(fs) for _, _, fs in os.walk(LOCAL_IMG_DIR)) if os.path.exists(LOCAL_IMG_DIR) else 0
print(f'  ✅  images/                           {n_img:>8,} files [SSD]')
print('='*60)
print('🎉 Dataset sẵn sàng — Tiếp tục Bước 4!')


## 🤝 Bước 4 — Positive Pairs (có Checkpoint)
Tự động bỏ qua nếu đã xử lý xong ở session trước.


In [ ]:
import pandas as pd, numpy as np, time, warnings, json
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════
# CATEGORY_MAPPING: map product_type_name → nhãn tiếng Việt (16 nhóm)
# Dựa trên dữ liệu thực của H&M articles.csv, được map sang nhãn thân thiện
# ══════════════════════════════════════════════════════════════════════════
CATEGORY_MAPPING = {
    # ── 👔 Đồ mặc trên (Áo) ──────────────────────────────────────────────
    'Áo Sơ Mi'   : ['Shirt', 'Blouse'],
    'Áo Thun'    : ['T-shirt', 'Top', 'Vest top'],
    'Áo Len/Nỉ'  : ['Sweater', 'Hoodie', 'Cardigan'],
    'Áo Khoác'   : ['Jacket', 'Blazer', 'Coat'],
    # ── 👖 Đồ mặc dưới (Quần/Váy) ────────────────────────────────────────
    'Quần Dài'   : ['Trousers', 'Leggings/Tights'],
    'Quần Ngắn'  : ['Shorts'],
    'Chân Váy'   : ['Skirt'],
    # ── 👗 Đồ Liền Thân ──────────────────────────────────────────────────
    'Váy Liền'   : ['Dress'],
    'Đồ Bộ'      : ['Garment Set', 'Jumpsuit/Playsuit'],
    # ── 👟 Giày Dép ───────────────────────────────────────────────────────
    'Sneakers'   : ['Sneakers'],
    'Giày Da/Boot': ['Boots'],
    'Sandal/Dép' : ['Sandals'],
    # ── 👜 Phụ Kiện ───────────────────────────────────────────────────────
    'Túi Xách'   : ['Bag'],
    'Mũ/Nón'     : ['Hat/beanie', 'Cap/peaked'],
    'Trang Sức'  : ['Earring', 'Necklace', 'Belt', 'Scarf',
                    'Other accessories', 'Sunglasses'],
    # ── 🩱 Đồ Lót / Đồ Mặc Nhà (không tham gia pairing outfit chính) ────
    'Đồ Lót'     : ['Underwear bottom', 'Bra', 'Swimwear bottom', 'Bikini top',
                    'Swimsuit', 'Bodysuit', 'Pyjama set', 'Underwear Tights',
                    'Underwear body'],
}

# Các nhóm tham gia tạo outfit (bỏ Đồ Lót ra vì không liên quan pairing)
OUTFIT_ROLES = [r for r in CATEGORY_MAPPING if r != 'Đồ Lót']

# ══════════════════════════════════════════════════════════════════════════
# ROLE_PAIRS: Tập luật phối đồ hợp lệ (bao phủ đầy đủ phong cách VN)
# A=Công Sở, B=Đường Phố/Casual, C=Nhiệt Đới/Hè, D=Mùa Đông, E=Phụ Kiện
# ══════════════════════════════════════════════════════════════════════════
ROLE_PAIRS = [
    # A. Phong cách Công Sở / Lịch sự ─────────────────────────────────────
    ('Áo Sơ Mi',    'Quần Dài'),
    ('Áo Sơ Mi',    'Chân Váy'),
    ('Áo Khoác',    'Áo Sơ Mi'),    # Khoác blazer ra ngoài sơ mi
    ('Áo Khoác',    'Váy Liền'),
    ('Áo Khoác',    'Quần Dài'),
    ('Giày Da/Boot','Quần Dài'),
    ('Giày Da/Boot','Chân Váy'),
    ('Giày Da/Boot','Váy Liền'),
    # B. Phong cách Đường Phố / Casual ─────────────────────────────────────
    ('Áo Thun',     'Quần Dài'),    # Áo thun + quần jeans/ống rộng
    ('Áo Thun',     'Quần Ngắn'),   # Mát mẻ mùa hè
    ('Áo Thun',     'Chân Váy'),
    ('Áo Sơ Mi',    'Quần Ngắn'),   # Sơ mi khoác ngoài rất phổ biến
    ('Sneakers',    'Quần Dài'),
    ('Sneakers',    'Quần Ngắn'),
    ('Sneakers',    'Váy Liền'),    # Váy + Sneaker trắng đang là trend
    ('Sneakers',    'Chân Váy'),
    ('Áo Len/Nỉ',   'Quần Dài'),
    ('Áo Len/Nỉ',   'Chân Váy'),
    ('Áo Len/Nỉ',   'Quần Ngắn'),   # Len nhẹ mùa thu
    # C. Phong cách Nhiệt Đới / Hè (đặc trưng VN nóng ẩm) ─────────────────
    ('Váy Liền',    'Sandal/Dép'),
    ('Quần Ngắn',   'Sandal/Dép'),
    ('Áo Thun',     'Sandal/Dép'),
    ('Chân Váy',    'Sandal/Dép'),
    # D. Phong cách Mùa Đông (Hà Nội / Đà Lạt) ───────────────────────────
    ('Áo Khoác',    'Áo Len/Nỉ'),   # Mặc len bên trong, khoác ngoài
    ('Giày Da/Boot','Áo Len/Nỉ'),
    ('Sneakers',    'Áo Len/Nỉ'),
    # E. Phụ Kiện đi kèm tổng hợp ─────────────────────────────────────────
    ('Túi Xách',    'Áo Sơ Mi'),
    ('Túi Xách',    'Váy Liền'),
    ('Túi Xách',    'Áo Thun'),
    ('Túi Xách',    'Đồ Bộ'),
    ('Trang Sức',   'Váy Liền'),
    ('Trang Sức',   'Áo Sơ Mi'),
    ('Mũ/Nón',      'Áo Thun'),
    ('Mũ/Nón',      'Áo Sơ Mi'),
    # F. Đồ Bộ & Jumpsuit ───────────────────────────────────────────────────
    ('Đồ Bộ',       'Sandal/Dép'),
    ('Đồ Bộ',       'Sneakers'),
]

print('📖 Đọc articles.csv...')
articles = pd.read_csv('articles.csv', dtype={'article_id': str})
role_map = {}
for role, types in CATEGORY_MAPPING.items():
    mask = articles['product_type_name'].isin(types)
    for aid in articles.loc[mask, 'article_id'].tolist():
        role_map[aid] = role

articles['role'] = articles['article_id'].map(role_map)
articles = articles[articles['role'].notna()].copy()
aid_to_role  = articles.set_index('article_id')['role'].to_dict()
role_to_ids  = articles.groupby('role')['article_id'].apply(list).to_dict()

print(f'✅ Valid articles: {len(articles):,}')
print(articles['role'].value_counts().to_string())


In [ ]:
import os
# ── Auto-setup nếu biến Drive chưa được định nghĩa (kernel restart) ────
if 'CKPT_DIR' not in dir():
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    DRIVE_DIR   = '/content/drive/MyDrive/Virtual_Stylist_application'
    os.chdir(DRIVE_DIR)
    CKPT_DIR    = os.path.join(DRIVE_DIR, 'checkpoints')
    OUTPUTS_DIR = '/content/outputs'
    os.makedirs(CKPT_DIR,    exist_ok=True)
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    def ckpt(name): return os.path.join(CKPT_DIR, name)
    def done(name): return os.path.exists(ckpt(name))
    def skip(name): print(f'⏭️  [{name}] đã có trên Drive — bỏ qua!')
    print('🔄 Drive đã được mount tự động!')

from tqdm.notebook import tqdm
from itertools import combinations

POS_CKPT = ckpt('positive_pairs.parquet')

if done('positive_pairs.parquet'):
    skip('positive_pairs.parquet')
    pos_df = pd.read_parquet(POS_CKPT)
    print(f'✅ Loaded {len(pos_df):,} positive pairs từ Drive.')
else:
    print('📖 Đọc toàn bộ transactions...')
    t0 = time.time()
    txn = pd.read_csv('transactions_train.csv',
                      dtype={'article_id': str},
                      usecols=['customer_id','article_id','t_dat'])
    txn = txn[txn['article_id'].isin(articles['article_id'])].copy()
    print(f'✅ Transactions loaded: {len(txn):,}  ({time.time()-t0:.1f}s)')

    print('\n🔗 Xây dựng Positive pairs từ baskets...')
    pos_pairs = []
    grouped = txn.groupby(['customer_id','t_dat'])['article_id'].apply(list)

    ROLE_PAIR_SET = set(ROLE_PAIRS) | {(b,a) for a,b in ROLE_PAIRS}

    for (cid, dat), items in tqdm(grouped.items(), desc='Baskets', mininterval=5):
        items_with_role = [(a, aid_to_role[a]) for a in set(items) if a in aid_to_role]
        if len(items_with_role) < 2:
            continue
        for (a1,r1),(a2,r2) in combinations(items_with_role, 2):
            if (r1,r2) in ROLE_PAIR_SET:
                if r1 > r2: a1,a2,r1,r2 = a2,a1,r2,r1
                pos_pairs.append((a1,r1,a2,r2,1))

    pos_df = pd.DataFrame(pos_pairs, columns=['a1','r1','a2','r2','label']).drop_duplicates()
    del pos_pairs; import gc; gc.collect()
    # Cap at 1.5M pairs (stratified) to avoid OOM on Colab 12GB RAM
    MAX_PAIRS = 1_500_000
    if len(pos_df) > MAX_PAIRS:
        n_roles = pos_df.groupby(['r1','r2']).ngroups
        per_role = MAX_PAIRS // n_roles
        pos_df = (pos_df.groupby(['r1','r2'], group_keys=False)
                  .apply(lambda x: x.sample(min(len(x), per_role), random_state=42))
                  .sample(frac=1, random_state=42).reset_index(drop=True))
        print(f'Sampled to {len(pos_df):,} pairs (stratified, {n_roles} role combos)')
    pos_df.to_parquet(POS_CKPT, index=False)
    print(f'Positive pairs saved: {len(pos_df):,}')
    print(pos_df.groupby(['r1','r2']).size().to_string())


## 🔀 Bước 5 — Negative Sampling (Nhanh + Checkpoint)
Dùng chiến thuật **Vectorised Sampling** — nhanh hơn vòng lặp while 50–100 lần.
Chấp nhận ~1% collision (không ảnh hưởng đáng kể đến chất lượng)


In [ ]:
import os
# ── Auto-setup nếu biến Drive chưa được định nghĩa (kernel restart) ────
if 'CKPT_DIR' not in dir():
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    DRIVE_DIR   = '/content/drive/MyDrive/Virtual_Stylist_application'
    os.chdir(DRIVE_DIR)
    CKPT_DIR    = os.path.join(DRIVE_DIR, 'checkpoints')
    OUTPUTS_DIR = '/content/outputs'
    os.makedirs(CKPT_DIR,    exist_ok=True)
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    def ckpt(name): return os.path.join(CKPT_DIR, name)
    def done(name): return os.path.exists(ckpt(name))
    def skip(name): print(f'⏭️  [{name}] đã có trên Drive — bỏ qua!')
    print('🔄 Drive đã được mount tự động!')

import numpy as np, pandas as pd

ROLE_PAIRS = [
    # A. Công Sở / Lịch sự
    ('Áo Sơ Mi','Quần Dài'), ('Áo Sơ Mi','Chân Váy'),
    ('Áo Khoác','Áo Sơ Mi'), ('Áo Khoác','Váy Liền'), ('Áo Khoác','Quần Dài'),
    ('Giày Da/Boot','Quần Dài'), ('Giày Da/Boot','Chân Váy'), ('Giày Da/Boot','Váy Liền'),
    # B. Đường Phố / Casual
    ('Áo Thun','Quần Dài'), ('Áo Thun','Quần Ngắn'), ('Áo Thun','Chân Váy'),
    ('Áo Sơ Mi','Quần Ngắn'),
    ('Sneakers','Quần Dài'), ('Sneakers','Quần Ngắn'), ('Sneakers','Váy Liền'), ('Sneakers','Chân Váy'),
    ('Áo Len/Nỉ','Quần Dài'), ('Áo Len/Nỉ','Chân Váy'), ('Áo Len/Nỉ','Quần Ngắn'),
    # C. Nhiệt Đới / Hè
    ('Váy Liền','Sandal/Dép'), ('Quần Ngắn','Sandal/Dép'),
    ('Áo Thun','Sandal/Dép'), ('Chân Váy','Sandal/Dép'),
    # D. Mùa Đông
    ('Áo Khoác','Áo Len/Nỉ'), ('Giày Da/Boot','Áo Len/Nỉ'), ('Sneakers','Áo Len/Nỉ'),
    # E. Phụ Kiện
    ('Túi Xách','Áo Sơ Mi'), ('Túi Xách','Váy Liền'), ('Túi Xách','Áo Thun'), ('Túi Xách','Đồ Bộ'),
    ('Trang Sức','Váy Liền'), ('Trang Sức','Áo Sơ Mi'),
    ('Mũ/Nón','Áo Thun'), ('Mũ/Nón','Áo Sơ Mi'),
    # F. Đồ Bộ & Jumpsuit
    ('Đồ Bộ','Sandal/Dép'), ('Đồ Bộ','Sneakers'),
]


# ── Auto-reload pos_df ────────────────────────────────────────────────────
if 'pos_df' not in dir():
    _p = ckpt('positive_pairs.parquet')
    if not os.path.exists(_p): raise RuntimeError('Chua co positive_pairs.parquet! Chay Buoc 4 truoc.')
    pos_df = pd.read_parquet(_p)
    print(f'pos_df reloaded: {len(pos_df):,} rows')

# ── Auto-reload role_to_ids ─────────────────────────────────────────────
if 'role_to_ids' not in dir():
    _cat = {
        'Áo Sơ Mi':['Shirt','Blouse'], 'Áo Thun':['T-shirt','Top','Vest top'],
        'Áo Len/Nỉ':['Sweater','Hoodie','Cardigan'], 'Áo Khoác':['Jacket','Blazer','Coat'],
        'Quần Dài':['Trousers','Leggings/Tights'], 'Quần Ngắn':['Shorts'],
        'Chân Váy':['Skirt'], 'Váy Liền':['Dress'], 'Đồ Bộ':['Garment Set','Jumpsuit/Playsuit'],
        'Sneakers':['Sneakers'], 'Giày Da/Boot':['Boots'], 'Sandal/Dép':['Sandals'],
        'Túi Xách':['Bag'], 'Mũ/Nón':['Hat/beanie','Cap/peaked'],
        'Trang Sức':['Earring','Necklace','Belt','Scarf','Other accessories','Sunglasses'],
    }
    _arts = pd.read_csv('articles.csv', dtype={'article_id': str})
    _rm = {}
    for _r, _types in _cat.items():
        for _a in _arts.loc[_arts['product_type_name'].isin(_types),'article_id'].tolist(): _rm[_a] = _r
    _arts['role'] = _arts['article_id'].map(_rm)
    role_to_ids = _arts[_arts['role'].notna()].groupby('role')['article_id'].apply(list).to_dict()
    print('role_to_ids reloaded OK (16 nhãn chi tiết)')


ALL_CKPT = ckpt('all_pairs.parquet')
if done('all_pairs.parquet'):
    skip('all_pairs.parquet')
    all_pairs = pd.read_parquet(ALL_CKPT)
    print(f'Loaded {len(all_pairs):,} pairs from Drive.')
    print(all_pairs['label'].value_counts())
else:
    # Guard: resample pos_df neu checkpoint cu vuot nguong 1.5M rows
    MAX_PAIRS = 1_500_000
    if len(pos_df) > MAX_PAIRS:
        print(f'WARNING: pos_df {len(pos_df):,} rows > {MAX_PAIRS:,} - dang resample...')
        n_roles = pos_df.groupby(['r1','r2']).ngroups
        per_role = MAX_PAIRS // n_roles
        pos_df = (pos_df.groupby(['r1','r2'], group_keys=False)
                  .apply(lambda x: x.sample(min(len(x), per_role), random_state=42))
                  .sample(frac=1, random_state=42).reset_index(drop=True))
        pos_df.to_parquet(ckpt('positive_pairs.parquet'), index=False)
        print(f'Da resample va ghi de checkpoint: {len(pos_df):,} rows')
    import gc; np.random.seed(42)
    target_neg = len(pos_df)
    print(f'Generating {target_neg:,} negative pairs...')
    # Dung integer hash thay vi string key -> tiet kiem ~60% RAM
    _M = 10**7
    def _h(a): return int(a.lstrip('0') or '0') % _M
    pos_set = set(_h(a)*_M + _h(b) for a,b in zip(pos_df['a1'], pos_df['a2']))
    pos_set |= set(_h(a)*_M + _h(b) for a,b in zip(pos_df['a2'], pos_df['a1']))
    print(f'pos_set (int-hash): {len(pos_set):,} entries')
    per_pair   = int(target_neg * 1.2) // len(ROLE_PAIRS) + 1
    neg_chunks = []
    for r1, r2 in ROLE_PAIRS:
        ids1 = np.array(role_to_ids[r1])
        ids2 = np.array(role_to_ids[r2])
        a1s  = np.random.choice(ids1, per_pair, replace=True)
        a2s  = np.random.choice(ids2, per_pair, replace=True)
        hashes = np.array([_h(a)*_M + _h(b) for a,b in zip(a1s, a2s)])
        keep = np.array([h not in pos_set for h in hashes])
        neg_chunks.append(pd.DataFrame({'a1':a1s[keep],'r1':r1,'a2':a2s[keep],'r2':r2,'label':0}))
        print(f'  {r1}/{r2}: {keep.sum():,} kept')
    del pos_set; gc.collect()
    neg_df    = pd.concat(neg_chunks, ignore_index=True)
    del neg_chunks; gc.collect()
    neg_df    = neg_df.sample(n=min(target_neg, len(neg_df)), random_state=42)
    all_pairs = pd.concat([pos_df, neg_df]).sample(frac=1, random_state=42).reset_index(drop=True)
    del neg_df; gc.collect()
    all_pairs.to_parquet(ALL_CKPT, index=False)
    print(f'Total pairs: {len(all_pairs):,} saved!')
    print(all_pairs['label'].value_counts())


## 🖼️ Bước 6 — Extract CLIP Embeddings (có Checkpoint)

In [ ]:
import os
# ── Auto-setup nếu biến Drive chưa được định nghĩa (kernel restart) ────
if 'CKPT_DIR' not in dir():
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    DRIVE_DIR   = '/content/drive/MyDrive/Virtual_Stylist_application'
    os.chdir(DRIVE_DIR)
    CKPT_DIR    = os.path.join(DRIVE_DIR, 'checkpoints')
    OUTPUTS_DIR = '/content/outputs'
    os.makedirs(CKPT_DIR,    exist_ok=True)
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    def ckpt(name): return os.path.join(CKPT_DIR, name)
    def done(name): return os.path.exists(ckpt(name))
    def skip(name): print(f'⏭️  [{name}] đã có trên Drive — bỏ qua!')
    print('🔄 Drive đã được mount tự động!')

import pickle, torch, torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from PIL import Image, UnidentifiedImageError

EMBED_DIM    = 512
CLIP_BATCH   = 64
EMB_CKPT     = ckpt('clip_embeddings.pkl')

# ── Tim thu muc anh (UU TIEN SSD CUC BO TRUOC) ────────────────────────────────
for _img_candidate in ['/content/images', 'images', 'h-and-m-personalized-fashion-recommendations/images']:
    if os.path.isdir(_img_candidate):
        IMAGE_DIR = _img_candidate
        break
else:
    raise RuntimeError(
        'Khong tim thay thu muc anh! Dataset chua duoc giai nen dung cach.\n'
        "Kiem tra: !ls -la '/content/' va tim thu muc 'images'"
    )
print(f'IMAGE_DIR: {IMAGE_DIR}')
_n_img = sum(len(fs) for _, _, fs in os.walk(IMAGE_DIR))
print(f'So luong anh tim thay: {_n_img:,}')
if _n_img == 0:
    raise RuntimeError(f'Thu muc {IMAGE_DIR} rong! Chay lai Buoc 3 de giai nen dataset.')

# ── Auto-reload articles neu bien bi mat ──────────────────────────────────
if 'articles' not in dir() or len(articles) == 0:
    print('Reloading articles.csv (16 nhãn chi tiết)...')
    _arts_tmp = pd.read_csv('articles.csv', dtype={'article_id': str})
    _cat_tmp  = {
        'Áo Sơ Mi'   : ['Shirt', 'Blouse'],
        'Áo Thun'    : ['T-shirt', 'Top', 'Vest top'],
        'Áo Len/Nỉ'  : ['Sweater', 'Hoodie', 'Cardigan'],
        'Áo Khoác'   : ['Jacket', 'Blazer', 'Coat'],
        'Quần Dài'   : ['Trousers', 'Leggings/Tights'],
        'Quần Ngắn'  : ['Shorts'],
        'Chân Váy'   : ['Skirt'],
        'Váy Liền'   : ['Dress'],
        'Đồ Bộ'      : ['Garment Set', 'Jumpsuit/Playsuit'],
        'Sneakers'   : ['Sneakers'],
        'Giày Da/Boot': ['Boots'],
        'Sandal/Dép' : ['Sandals'],
        'Túi Xách'   : ['Bag'],
        'Mũ/Nón'     : ['Hat/beanie', 'Cap/peaked'],
        'Trang Sức'  : ['Earring', 'Necklace', 'Belt', 'Scarf',
                        'Other accessories', 'Sunglasses'],
    }
    _rm_tmp = {}
    for _r, _types in _cat_tmp.items():
        for _a in _arts_tmp.loc[_arts_tmp['product_type_name'].isin(_types), 'article_id'].tolist():
            _rm_tmp[_a] = _r
    _arts_tmp['role'] = _arts_tmp['article_id'].map(_rm_tmp)
    articles = _arts_tmp[_arts_tmp['role'].notna()].copy()
    del _arts_tmp, _rm_tmp, _cat_tmp
    print(f'articles reloaded: {len(articles):,}')


# ── Auto-reload all_pairs ─────────────────────────────────────────────────
if 'all_pairs' not in dir() or len(all_pairs) == 0:
    _ap = ckpt('all_pairs.parquet')
    if not os.path.exists(_ap):
        raise RuntimeError('Chua co all_pairs.parquet! Chay Buoc 5 truoc.')
    all_pairs = pd.read_parquet(_ap)
    print(f'all_pairs reloaded: {len(all_pairs):,} rows')

# ── Load hoac extract embeddings ──────────────────────────────────────────
_need_extract = True
if done('clip_embeddings.pkl'):
    skip('clip_embeddings.pkl')
    with open(EMB_CKPT, 'rb') as f:
        id2emb = pickle.load(f)
    if len(id2emb) == 0:
        print('WARN: clip_embeddings.pkl corrupt (0 embeddings) — se extract lai.')
        os.remove(EMB_CKPT)
    else:
        print(f'Loaded {len(id2emb):,} embeddings from Drive.')
        _need_extract = False

if _need_extract:
    CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'
    print(f'Loading {CLIP_MODEL_ID}...')
    clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
    clip_model.eval()

    unique_ids = articles['article_id'].tolist()
    id2emb = {}

    @torch.no_grad()
    def clip_batch_fn(ids):
        imgs, valid_ids = [], []
        for aid in ids:
            _padded = aid.zfill(10)  # H&M: '108775015' -> '0108775015'
            _folder = _padded[:3]    # folder = first 3 chars: '010'
            path   = os.path.join(IMAGE_DIR, _folder, f'{_padded}.jpg')
            try:
                imgs.append(Image.open(path).convert('RGB'))
                valid_ids.append(aid)
            except (FileNotFoundError, UnidentifiedImageError):
                pass
        if not imgs: return
        inputs = clip_processor(images=imgs, return_tensors='pt').to(device)
        # Dung vision_model + visual_projection thay get_image_features()
        # de tranh AttributeError voi cac phien ban transformers moi hon
        vision_out = clip_model.vision_model(pixel_values=inputs['pixel_values'])
        feats = clip_model.visual_projection(vision_out.pooler_output)  # (B, 512)
        feats  = F.normalize(feats, p=2, dim=-1).cpu().numpy()
        for aid, emb in zip(valid_ids, feats):
            id2emb[aid] = emb

    from tqdm.notebook import tqdm
    for i in tqdm(range(0, len(unique_ids), CLIP_BATCH), desc='CLIP extract'):
        clip_batch_fn(unique_ids[i:i+CLIP_BATCH])

    if len(id2emb) == 0:
        raise RuntimeError(
            f'Extract xong nhung id2emb van rong!\n'
            f'Kiem tra IMAGE_DIR={IMAGE_DIR} co dung cau truc 0xxx/xxxxxxxx.jpg khong.\n'
            f"Thu: !ls '{IMAGE_DIR}' | head -5"
        )

    with open(EMB_CKPT, 'wb') as f:
        pickle.dump(id2emb, f)
    print(f'Extracted {len(id2emb):,} embeddings — saved to Drive!')

# ── Filter pairs ──────────────────────────────────────────────────────────
all_pairs = all_pairs[
    all_pairs['a1'].isin(id2emb) & all_pairs['a2'].isin(id2emb)
].reset_index(drop=True)
print(f'Pairs hop le (co embedding): {len(all_pairs):,}')
if len(all_pairs) == 0:
    raise RuntimeError('Sau filter, all_pairs rong! id2emb co the khong khop voi article_id trong all_pairs.')


## 🧠 Bước 7 — Định nghĩa & Huấn luyện Siamese Model
**Model checkpoint được lưu lên Drive sau mỗi epoch** — session mới sẽ load và train tiếp từ epoch cuối.


In [ ]:
import os
# ── Auto-setup nếu biến Drive chưa được định nghĩa (kernel restart) ────
if 'CKPT_DIR' not in dir():
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    DRIVE_DIR   = '/content/drive/MyDrive/Virtual_Stylist_application'
    os.chdir(DRIVE_DIR)
    CKPT_DIR    = os.path.join(DRIVE_DIR, 'checkpoints')
    OUTPUTS_DIR = '/content/outputs'
    os.makedirs(CKPT_DIR,    exist_ok=True)
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    def ckpt(name): return os.path.join(CKPT_DIR, name)
    def done(name): return os.path.exists(ckpt(name))
    def skip(name): print(f'⏭️  [{name}] đã có trên Drive — bỏ qua!')
    print('🔄 Drive đã được mount tự động!')

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# ── Model definition ──────────────────────────────────────────────────────
class CompatibilityMLP(nn.Module):
    def __init__(self, emb_dim=512, hidden=256):
        super().__init__()
        in_dim = emb_dim * 3
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1), nn.Sigmoid(),
        )
    def forward(self, a, b):
        return self.net(torch.cat([a, b, torch.abs(a-b)], dim=1)).squeeze(-1)

# ── Dataset ───────────────────────────────────────────────────────────────
class PairDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return (torch.tensor(id2emb[row.a1], dtype=torch.float32),
                torch.tensor(id2emb[row.a2], dtype=torch.float32),
                torch.tensor(row.label, dtype=torch.float32))

# ── Auto-reload all_pairs neu bien bi mat sau kernel restart ──────────────
import pickle
if 'all_pairs' not in dir() or len(all_pairs) == 0:
    _ap = ckpt('all_pairs.parquet')
    if not os.path.exists(_ap):
        raise RuntimeError('Chua co all_pairs.parquet! Chay Buoc 5 truoc.')
    all_pairs = pd.read_parquet(_ap)
    print(f'all_pairs reloaded: {len(all_pairs):,} rows | label: {all_pairs["label"].value_counts().to_dict()}')

if len(all_pairs) == 0:
    raise RuntimeError('all_pairs rong! Xoa checkpoint all_pairs.parquet va chay lai Buoc 5.')

# ── Auto-reload id2emb neu bien bi mat ────────────────────────────────────
if 'id2emb' not in dir() or len(id2emb) == 0:
    _ep = ckpt('clip_embeddings.pkl')
    if not os.path.exists(_ep):
        raise RuntimeError('Chua co clip_embeddings.pkl! Chay Buoc 6 truoc.')
    with open(_ep, 'rb') as _f:
        id2emb = pickle.load(_f)
    print(f'id2emb reloaded: {len(id2emb):,} embeddings')
    # Neu pkl bi corrupt (load ve rong), xoa va yeu cau chay lai Buoc 6
    if len(id2emb) == 0:
        os.remove(_ep)
        raise RuntimeError(
            'clip_embeddings.pkl bi corrupt (0 embeddings) — da xoa file loi.\n'
            'Vui long chay lai Buoc 6 de extract lai embeddings.'
        )
    # NOTE: Khong can filter lai all_pairs — checkpoint da duoc filter o Buoc 6 truoc khi save.

EMBED_DIM = 512  # dam bao bien ton tai cho CompatibilityMLP

train_df, val_df = train_test_split(all_pairs, test_size=0.1,
                                     stratify=all_pairs['label'], random_state=42)
BATCH_SIZE   = 512
train_loader = DataLoader(PairDataset(train_df), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(PairDataset(val_df),   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

# ── Load checkpoint nếu đang resume ────────────────────────────────────────
TRAIN_CKPT  = ckpt('siamese_latest.pt')
EPOCHS      = 20
LR          = 3e-4

model     = CompatibilityMLP(EMBED_DIM, 256).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCELoss()

start_epoch = 1
best_auc    = 0.0
history     = {'train_loss':[], 'val_loss':[], 'val_auc':[]}

if os.path.exists(TRAIN_CKPT):
    ckpt_data   = torch.load(TRAIN_CKPT, map_location=device)
    model.load_state_dict(ckpt_data['model_state'])
    optimizer.load_state_dict(ckpt_data['optim_state'])
    scheduler.load_state_dict(ckpt_data['sched_state'])
    start_epoch = ckpt_data['epoch'] + 1
    best_auc    = ckpt_data['best_auc']
    history     = ckpt_data.get('history', history)
    print(f'🔄 RESUME từ epoch {start_epoch} | Best AUC so far: {best_auc:.4f}')
else:
    print('🆕 Bắt đầu training từ đầu...')

# ── Training loop ─────────────────────────────────────────────────────────
best_state = model.state_dict()

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for emb_a, emb_b, labels in train_loader:
        emb_a, emb_b, labels = emb_a.to(device), emb_b.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(emb_a, emb_b), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
    train_loss /= len(train_df)

    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for emb_a, emb_b, labels in val_loader:
            emb_a, emb_b, labels = emb_a.to(device), emb_b.to(device), labels.to(device)
            preds = model(emb_a, emb_b)
            val_loss += criterion(preds, labels).item() * len(labels)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    val_loss /= len(val_df)
    val_auc   = roc_auc_score(all_labels, all_preds)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    is_best = val_auc > best_auc
    if is_best:
        best_auc   = val_auc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    # ── Lưu checkpoint sau mỗi epoch ─────────────────────────────────────
    torch.save({
        'epoch'      : epoch,
        'model_state': model.state_dict(),
        'optim_state': optimizer.state_dict(),
        'sched_state': scheduler.state_dict(),
        'best_auc'   : best_auc,
        'history'    : history,
    }, TRAIN_CKPT)

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'TLoss: {train_loss:.4f} | '
          f'VLoss: {val_loss:.4f} | '
          f'AUC: {val_auc:.4f}'
          f' 🏆 saved!' if is_best else '')

model.load_state_dict(best_state)
print(f'\n✅ Training DONE | Best Val AUC: {best_auc:.4f}')


## 🎨 Bước 8 — Demo Inference: Phối đồ thực tế

In [ ]:
articles_indexed = articles.set_index('article_id')

# Tim IMAGE_DIR voi fallback (uu tien cuc bo)
for _ic in ['/content/images', 'images', 'h-and-m-personalized-fashion-recommendations/images']:
    if os.path.isdir(_ic): IMAGE_DIR = _ic; break

def get_image(article_id):
    _padded = str(article_id).zfill(10)
    path   = os.path.join(IMAGE_DIR, _padded[:3], f'{_padded}.jpg')
    try: return Image.open(path).convert('RGB')
    except: return None

@torch.no_grad()
def suggest_outfits(query_id, top_k=4, pool=1000):
    query_role = aid_to_role.get(query_id)
    query_emb  = torch.tensor(id2emb[query_id], dtype=torch.float32).unsqueeze(0).to(device)
    target_roles = set()
    for r1, r2 in ROLE_PAIRS:
        if r1 == query_role: target_roles.add(r2)
        elif r2 == query_role: target_roles.add(r1)
    target_roles = list(target_roles)

    results_by_role = {}
    for role in target_roles:
        candidates = [a for a in role_to_ids.get(role, []) if a in id2emb]
        if len(candidates) > pool:
            candidates = np.random.choice(candidates, pool, replace=False).tolist()
        if not candidates: continue
        embs   = torch.tensor(np.stack([id2emb[a] for a in candidates]), dtype=torch.float32).to(device)
        q_rep  = query_emb.expand(len(candidates), -1)
        model.eval()
        scores = model(q_rep, embs).cpu().numpy()
        top_idx = np.argsort(scores)[::-1][:top_k]
        results_by_role[role] = [(candidates[i], float(scores[i])) for i in top_idx]

    palette = ['#F28E2B', '#4E79A7', '#59A14F', '#E15759', '#F1CE63', '#B07AA1', '#FF9D9A', '#9D7660', '#D37295']
    ROLE_COLORS = {r: palette[i % len(palette)] for i, r in enumerate(target_roles)}
    n_roles = len(results_by_role)
    fig, axes = plt.subplots(n_roles, top_k+1, figsize=(3.2*(top_k+1), 3.5*n_roles), facecolor='#1a1a2e')
    if n_roles == 1: axes = [axes]

    query_info = articles_indexed.loc[query_id]
    fig.suptitle(f'🎨 Outfit for: {query_info.get("prod_name",query_id)[:40]}\nRole: {query_role}',
                 color='white', fontsize=13, fontweight='bold', y=1.01)

    for row_idx, (role, items) in enumerate(results_by_role.items()):
        row_axes = axes[row_idx]
        ax0 = row_axes[0]
        img = get_image(query_id)
        if img: ax0.imshow(img)
        ax0.set_title(f'QUERY\n{query_role}', color='#F28E2B', fontsize=9, fontweight='bold')
        ax0.axis('off'); ax0.set_facecolor('#16213e')
        for col_idx, (aid, score) in enumerate(items):
            ax  = row_axes[col_idx+1]
            img = get_image(aid)
            if img: ax.imshow(img)
            try: name = articles_indexed.loc[aid,'prod_name'][:20]
            except: name = aid
            ax.set_title(f'#{col_idx+1}  {score:.1%}\n{name}\n{role}',
                         color=ROLE_COLORS.get(role,'white'), fontsize=7.5)
            ax.axis('off'); ax.set_facecolor('#16213e')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, 'outfit_suggestion.png'), dpi=150, bbox_inches='tight')
    plt.show()
    return results_by_role

from PIL import Image
import random
demo_id = random.choice([a for a in role_to_ids.get('Áo Thun', list(role_to_ids.values())[0]) if a in id2emb])
print(f'Demo với article: {demo_id}')
suggest_outfits(demo_id, top_k=4, pool=1000)


## 💾 Bước 9 — Lưu Final Model & Tải Về Máy
Mô hình và các dữ liệu phụ trợ (embeddings, metadata) sẽ được lưu thẳng vào Drive. 
Nếu bạn muốn **tải xuống máy tính cá nhân**, hãy sử dụng đoạn mã ở ô bên dưới.

In [ ]:
import os
# ── Auto-setup nếu biến Drive chưa được định nghĩa (kernel restart) ────
if 'CKPT_DIR' not in dir():
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    DRIVE_DIR   = '/content/drive/MyDrive/Virtual_Stylist_application'
    os.chdir(DRIVE_DIR)
    CKPT_DIR    = os.path.join(DRIVE_DIR, 'checkpoints')
    OUTPUTS_DIR = '/content/outputs'
    os.makedirs(CKPT_DIR,    exist_ok=True)
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    def ckpt(name): return os.path.join(CKPT_DIR, name)
    def done(name): return os.path.exists(ckpt(name))
    def skip(name): print(f'⏭️  [{name}] đã có trên Drive — bỏ qua!')
    print('🔄 Drive đã được mount tự động!')

FINAL_MODEL = os.path.join(DRIVE_DIR, 'siamese_best.pt')
ROLE_FILE   = os.path.join(DRIVE_DIR, 'article_roles.json')

torch.save({'model_state': model.state_dict(),
            'emb_dim': EMBED_DIM, 'hidden': 256, 'best_auc': best_auc}, FINAL_MODEL)

with open(ROLE_FILE, 'w') as f:
    json.dump(aid_to_role, f)

import shutil
shutil.copy(os.path.join(OUTPUTS_DIR, 'outfit_suggestion.png'), DRIVE_DIR)

print('\n📤 Final artifacts trên Google Drive:')
for p in [FINAL_MODEL, EMB_CKPT, ROLE_FILE]:
    mb = os.path.getsize(p)/1e6
    print(f'   {os.path.basename(p):45s}  {mb:.1f} MB')
print(f'\n🎉 Hoàn thành! Best Val AUC = {best_auc:.4f}')


### ⬇️ Download files về máy tính (Tùy chọn)
Bỏ comment các dòng lệnh dưới đây để trình duyệt tự động tải file Model và Embeddings.

In [ ]:
from google.colab import files

# Tải file Model Weights
# files.download(FINAL_MODEL)

# Tải danh sách Categories/Roles
# files.download(ROLE_FILE)

# Tải CLIP Embeddings (Lưu ý: File ~200MB, sẽ hơi mất thời gian tải)
# files.download(EMB_CKPT)

print('Bỏ dấu \'#\' ở các dòng files.download() phía trên để tải file mong muốn về máy.')
